# I/O benchmark for large MuData

MuData supports **backed mode** (`mu.read_h5mu(path, backed="r")`) where count
matrices stay on disk and minibatches are loaded via h5py during training. This can
dramatically reduce RAM usage at the cost of slower I/O.

We benchmark three options:
1. **In-memory** (default) — fastest, but requires all data in RAM
2. **Backed from network lustre** — minimal RAM, but network I/O overhead
3. **Backed from local `/tmp`** — copy to local SSD first, then back from there

**Input**: `gastrula_to_pup_filtered_qc.h5mu` (cell-filtered by NB1d)

Run once on a high-memory node to decide which option to use in NB2 training.

In [ ]:
import os
import shutil
import time

import mudata as mu
import numpy as np
import psutil

DATA_DIR = "/nfs/team205/vk7/sanger_projects/large_data/gastrula_to_pup"

In [ ]:
h5mu_filt = os.path.join(DATA_DIR, "gastrula_to_pup_filtered_qc.h5mu")
print(f"Loading {h5mu_filt} in-memory...")
mdata = mu.read_h5mu(h5mu_filt)
print(f"RSS after load: {psutil.Process().memory_info().rss / 1e9:.1f} GB")
for mod_name in mdata.mod:
    print(f"  {mod_name}: {mdata[mod_name].shape}")

n_cells = mdata["rna"].n_obs
batch_idx = np.sort(np.random.choice(n_cells, 1024, replace=False))
N_READS = 100

# Option A: In-memory
t0 = time.time()
for _ in range(N_READS):
    for mod_name in mdata.mod:
        _ = mdata[mod_name].X[batch_idx]
t_mem = time.time() - t0

# Option B: Backed from lustre
mdata_lustre = mu.read_h5mu(h5mu_filt, backed="r")
t0 = time.time()
for _ in range(N_READS):
    for mod_name in mdata_lustre.mod:
        _ = mdata_lustre[mod_name].X[batch_idx]
t_lustre = time.time() - t0
mdata_lustre.file.close()
del mdata_lustre

# Option C: Backed from local /tmp
local_path = "/tmp/gastrula_to_pup_filtered_qc.h5mu"
print(f"\nCopying to {local_path}...")
shutil.copy2(h5mu_filt, local_path)
mdata_local = mu.read_h5mu(local_path, backed="r")
t0 = time.time()
for _ in range(N_READS):
    for mod_name in mdata_local.mod:
        _ = mdata_local[mod_name].X[batch_idx]
t_local = time.time() - t0
mdata_local.file.close()
del mdata_local
os.remove(local_path)

# Results
print(f"\n{'Option':<25} {'Time (s)':>10} {'Relative':>10}")
print("-" * 47)
print(f"{'In-memory':<25} {t_mem:10.2f} {'1.0x':>10}")
print(f"{'Backed (lustre)':<25} {t_lustre:10.2f} {f'{t_lustre / t_mem:.1f}x':>10}")
print(f"{'Backed (/tmp local)':<25} {t_local:10.2f} {f'{t_local / t_mem:.1f}x':>10}")

# Estimate full training time impact (25 epochs x ~N steps/epoch)
steps_per_epoch = n_cells // 1024
total_reads = 25 * steps_per_epoch
print(f"\nEstimated total training I/O overhead ({total_reads:,} reads):")
print(f"  In-memory: {total_reads * t_mem / N_READS / 3600:.1f} hours")
print(f"  Backed (lustre): {total_reads * t_lustre / N_READS / 3600:.1f} hours")
print(f"  Backed (/tmp): {total_reads * t_local / N_READS / 3600:.1f} hours")

## Recommendation

- If the node has sufficient RAM (>300GB free after model), use **in-memory** for fastest training.
- If RAM is limited, copy to `/tmp` and use **backed from /tmp** — much faster than network lustre.
- Avoid **backed from lustre** unless no other option — network I/O adds significant overhead.

Set the loading option in NB2 cell 3 accordingly.